# Generalized Agentic XAI Workflow: Appliances Energy Regression

This notebook applies the reusable tabular Agentic XAI framework to a real
regression dataset. Dataset-specific choices remain in the configuration cell.

Workflow:

`load data -> temporal split -> train model -> SHAP -> figures -> prompts -> recommendations -> refinement rounds`

Raw date columns are retained for audit and splitting. Optional cyclical calendar
features are generated generically so the same framework can use other dated
tabular datasets.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from matplotlib.backends.backend_pdf import PdfPages
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")


# ============================================================
# Dataset configuration: change this section for another dataset
# ============================================================

DATASET_PATH = Path("energydata_complete.csv")
TARGET_COLUMN = "Appliances"
TASK_TYPE = "regression"  # "regression" or "classification"
DOMAIN_CONTEXT = (
    "appliance energy consumption prediction using indoor temperature and "
    "humidity measurements, outdoor weather conditions, lighting, and time context"
)

# These columns remain available for audit/splitting but do not enter directly.
ID_COLUMNS = []
DATE_COLUMNS = ["date"]

# rv1 and rv2 are identical synthetic random variables in this public dataset.
DROP_COLUMNS = ["rv1", "rv2"]

# Generic calendar features derived from each configured date column.
GENERATE_DATE_FEATURES = True

# Optional outputs from a previous model. They are never used as model inputs.
REFERENCE_COLUMNS = []
REFERENCE_PREDICTION_COLUMN = None
REFERENCE_PROBABILITY_COLUMN = None
REFERENCE_IS_OUT_OF_SAMPLE = False

# The primary split is comparable with conventional tabular benchmarks.
# Round 1 then challenges it with grouped-day and future-period evaluation.
SPLIT_STRATEGY = "random"  # "random", "group", or "temporal"
GROUP_COLUMNS = []
TIME_COLUMN = "date"
RUN_DATE_SPLIT_STRESS_TESTS = True
DATE_GROUP_FREQUENCY = "D"

TEST_SIZE = 0.20
RANDOM_STATE = 42
POSITIVE_CLASS = 1
TOP_K_FEATURES = 20
SHAP_SAMPLE_SIZE = 1000
SHAP_APPROXIMATE = True
ROUND_ID = 0

# Paper-aligned artifact structure.
OUTPUT_DIR = Path("Evidence")
FIGURES_DIR = Path("Figures")
PROMPTS_DIR = Path("Prompts")
RECOMMENDATIONS_DIR = Path("Recommendations")

# Optional domain grouping affects only one summary figure.
FEATURE_GROUPS = {
    "indoor_temperature": [f"t{index}" for index in range(1, 10)],
    "indoor_humidity": [f"rh_{index}" for index in range(1, 10)],
    "outdoor_weather": [
        "t_out", "press_mm_hg", "rh_out", "windspeed", "visibility",
        "tdewpoint",
    ],
    "lighting": ["lights"],
    "time_context": ["date_"],
}

# Manual mode writes reproducible prompts and evidence-backed recommendation drafts.
LLM_PROVIDER = "manual"

MODEL_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_leaf": 1,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}


## Data Loading

The loader keeps the full audit table in `df`, while building `X_raw` only from
eligible model features. Target duplicates and prior-model outputs never enter `X_raw`.


In [2]:
def read_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {suffix}. Use CSV, Excel, or Parquet.")


def existing_columns(df, configured_columns, label):
    configured_columns = configured_columns or []
    existing = [column for column in configured_columns if column in df.columns]
    missing = sorted(set(configured_columns) - set(existing))
    if missing:
        print(f"Note: configured {label} columns not found and ignored: {missing}")
    return existing


def add_calendar_features(df, date_columns, enabled=True):
    """Create reusable cyclical calendar features without exposing raw timestamps."""
    if not enabled:
        return df, []

    added_columns = []
    for column in date_columns:
        parsed = pd.to_datetime(df[column], errors="coerce")
        if parsed.isna().any():
            invalid_count = int(parsed.isna().sum())
            raise ValueError(
                f"Date column {column!r} contains {invalid_count} invalid values."
            )

        minute_of_day = parsed.dt.hour * 60 + parsed.dt.minute
        day_position = parsed.dt.dayofweek + minute_of_day / (24 * 60)
        month_position = parsed.dt.month - 1

        features = {
            f"{column}_time_sin": np.sin(2 * np.pi * minute_of_day / (24 * 60)),
            f"{column}_time_cos": np.cos(2 * np.pi * minute_of_day / (24 * 60)),
            f"{column}_week_sin": np.sin(2 * np.pi * day_position / 7),
            f"{column}_week_cos": np.cos(2 * np.pi * day_position / 7),
            f"{column}_month_sin": np.sin(2 * np.pi * month_position / 12),
            f"{column}_month_cos": np.cos(2 * np.pi * month_position / 12),
            f"{column}_is_weekend": (parsed.dt.dayofweek >= 5).astype(int),
        }
        for feature_name, values in features.items():
            df[feature_name] = values
            added_columns.append(feature_name)

    return df, added_columns


def load_dataset(
    dataset_path,
    target_column,
    id_columns=None,
    date_columns=None,
    drop_columns=None,
    reference_columns=None,
    generate_date_features=True,
):
    df = read_table(dataset_path)
    if target_column not in df.columns:
        raise ValueError(f"TARGET_COLUMN={target_column!r} was not found.")

    id_columns = existing_columns(df, id_columns, "ID")
    date_columns = existing_columns(df, date_columns, "date")
    drop_columns = existing_columns(df, drop_columns, "drop")
    reference_columns = existing_columns(df, reference_columns, "reference")

    df = df.dropna(subset=[target_column]).copy()
    df, date_feature_columns = add_calendar_features(
        df,
        date_columns,
        enabled=generate_date_features,
    )

    excluded = list(
        dict.fromkeys(
            [target_column]
            + id_columns
            + date_columns
            + drop_columns
            + reference_columns
        )
    )

    X = df.drop(columns=excluded)
    y = df[target_column]
    reference_data = df[reference_columns].copy()
    audit_data = df[id_columns + date_columns].copy()

    if X.shape[1] == 0:
        raise ValueError("No model features remain after configured exclusions.")

    leakage = sorted(set(X.columns) & set(reference_columns + [target_column]))
    if leakage:
        raise RuntimeError(f"Leakage columns reached the feature matrix: {leakage}")

    return (
        df,
        X,
        y,
        reference_data,
        audit_data,
        excluded,
        date_feature_columns,
    )


(
    df,
    X_raw,
    y,
    reference_data,
    audit_data,
    excluded_columns,
    date_feature_columns,
) = load_dataset(
    DATASET_PATH,
    TARGET_COLUMN,
    id_columns=ID_COLUMNS,
    date_columns=DATE_COLUMNS,
    drop_columns=DROP_COLUMNS,
    reference_columns=REFERENCE_COLUMNS,
    generate_date_features=GENERATE_DATE_FEATURES,
)

print(f"Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Eligible model features: {X_raw.shape[1]}")
print(f"Target column: {TARGET_COLUMN}")
print(f"Task type: {TASK_TYPE}")
print(f"Columns retained outside training: {excluded_columns}")
print(f"Generated date features: {date_feature_columns}")
display(df.head())


Loaded dataset: 19,735 rows, 36 columns
Eligible model features: 32
Target column: Appliances
Task type: regression
Columns retained outside training: ['Appliances', 'date', 'rv1', 'rv2']
Generated date features: ['date_time_sin', 'date_time_cos', 'date_week_sin', 'date_week_cos', 'date_month_sin', 'date_month_cos', 'date_is_weekend']


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,Tdewpoint,rv1,rv2,date_time_sin,date_time_cos,date_week_sin,date_week_cos,date_month_sin,date_month_cos,date_is_weekend
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,5.3,13.275433,13.275433,-0.965926,-0.258819,0.593820,0.804598,0.0,1.0,0
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,5.2,18.606195,18.606195,-0.976296,-0.216440,0.598824,0.800881,0.0,1.0,0
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,5.1,28.642668,28.642668,-0.984808,-0.173648,0.603804,0.797133,0.0,1.0,0
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,5.0,45.410389,45.410389,-0.991445,-0.130526,0.608761,0.793353,0.0,1.0,0
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,4.9,10.084097,10.084097,-0.996195,-0.087156,0.613695,0.789543,0.0,1.0,0


## Preprocessing, Safe Splitting, And Model Training

Choose a split strategy in the configuration:

- `random`: general tabular baseline
- `group`: keeps the same site/entity out of both train and test
- `temporal`: trains on earlier records and tests on later records


In [3]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(X):
    numeric_columns = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_columns = [col for col in X.columns if col not in numeric_columns]

    transformers = []
    if numeric_columns:
        transformers.append(
            (
                "num",
                Pipeline([("imputer", SimpleImputer(strategy="median"))]),
                numeric_columns,
            )
        )
    if categorical_columns:
        transformers.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_one_hot_encoder()),
                    ]
                ),
                categorical_columns,
            )
        )
    return ColumnTransformer(transformers=transformers), numeric_columns, categorical_columns


def make_model(task_type, params):
    task_type = task_type.lower().strip()
    if task_type == "regression":
        return RandomForestRegressor(**params)
    if task_type == "classification":
        return RandomForestClassifier(**params)
    raise ValueError("TASK_TYPE must be 'regression' or 'classification'.")


def get_feature_names(preprocessor):
    return [
        name.replace("num__", "").replace("cat__", "")
        for name in preprocessor.get_feature_names_out()
    ]


def split_row_indices(df, y, task_type):
    strategy = SPLIT_STRATEGY.lower().strip()

    if strategy == "random":
        stratify = None
        if task_type.lower() == "classification":
            counts = y.value_counts(dropna=False)
            if len(counts) > 1 and counts.min() >= 2:
                stratify = y
        return train_test_split(
            df.index,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=stratify,
        )

    if strategy == "group":
        missing = [column for column in GROUP_COLUMNS if column not in df.columns]
        if missing:
            raise ValueError(f"GROUP_COLUMNS not found: {missing}")
        groups = df[GROUP_COLUMNS].astype(str).agg("||".join, axis=1)
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
        )
        train_positions, test_positions = next(splitter.split(df, y, groups))
        return df.index[train_positions], df.index[test_positions]

    if strategy == "temporal":
        if TIME_COLUMN not in df.columns:
            raise ValueError(f"TIME_COLUMN={TIME_COLUMN!r} was not found.")
        times = pd.to_datetime(df[TIME_COLUMN], errors="coerce")
        if times.isna().any():
            raise ValueError("Temporal split requires valid dates in every retained row.")
        ordered_indices = times.sort_values().index.to_numpy()
        cut = int(len(ordered_indices) * (1 - TEST_SIZE))
        return ordered_indices[:cut], ordered_indices[cut:]

    raise ValueError("SPLIT_STRATEGY must be 'random', 'group', or 'temporal'.")


train_indices, test_indices = split_row_indices(df, y, TASK_TYPE)
X_train_raw = X_raw.loc[train_indices]
X_test_raw = X_raw.loc[test_indices]
y_train = y.loc[train_indices]
y_test = y.loc[test_indices]

preprocessor, numeric_columns, categorical_columns = make_preprocessor(X_train_raw)
X_train_array = preprocessor.fit_transform(X_train_raw)
X_test_array = preprocessor.transform(X_test_raw)

feature_names = get_feature_names(preprocessor)
X_train = pd.DataFrame(X_train_array, columns=feature_names, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_array, columns=feature_names, index=X_test_raw.index)

model = make_model(TASK_TYPE, MODEL_PARAMS)
model.fit(X_train, y_train)

print(f"Split strategy: {SPLIT_STRATEGY}")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Numeric source columns: {len(numeric_columns)}")
print(f"Categorical source columns: {len(categorical_columns)}")
print(f"Transformed feature count: {X_train.shape[1]}")


Split strategy: random
Training rows: 15,788
Test rows: 3,947
Numeric source columns: 32
Categorical source columns: 0
Transformed feature count: 32


## Evaluation And Reference Comparison

Reference fields are evaluated only after the new model has predicted the test rows.
They are never supplied to preprocessing, training, or SHAP.


In [4]:
def classification_metrics(y_true, y_pred, probability=None):
    labels = pd.Series(y_true).dropna().unique().tolist()
    binary = len(labels) == 2 and POSITIVE_CLASS in labels
    average = "binary" if binary else "weighted"
    kwargs = {"average": average, "zero_division": 0}
    if binary:
        kwargs["pos_label"] = POSITIVE_CLASS

    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, **kwargs),
        "recall": recall_score(y_true, y_pred, **kwargs),
        "f1": f1_score(y_true, y_pred, **kwargs),
    }
    if probability is not None and binary:
        binary_truth = (pd.Series(y_true).to_numpy() == POSITIVE_CLASS).astype(int)
        result["roc_auc"] = roc_auc_score(binary_truth, probability)
    return result


def regression_metrics(y_true, y_pred):
    result = {
        "r2": r2_score(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": mean_absolute_error(y_true, y_pred),
    }
    try:
        result["mape"] = mean_absolute_percentage_error(y_true, y_pred)
    except Exception:
        result["mape"] = None
    return result


def get_positive_class_probability(model, X):
    if TASK_TYPE.lower() != "classification" or not hasattr(model, "predict_proba"):
        return None
    probabilities = model.predict_proba(X)
    classes = list(model.classes_)
    if len(classes) != 2:
        return None
    class_index = classes.index(POSITIVE_CLASS) if POSITIVE_CLASS in classes else 1
    return probabilities[:, class_index]


def evaluate_new_model(model, X_train, X_test, y_train, y_test):
    train_prediction = model.predict(X_train)
    test_prediction = model.predict(X_test)
    test_probability = get_positive_class_probability(model, X_test)

    if TASK_TYPE.lower() == "classification":
        train_metrics = classification_metrics(y_train, train_prediction)
        test_metrics = classification_metrics(y_test, test_prediction, test_probability)
        report = classification_report(
            y_test, test_prediction, output_dict=True, zero_division=0
        )
    else:
        train_metrics = regression_metrics(y_train, train_prediction)
        test_metrics = regression_metrics(y_test, test_prediction)
        report = None

    metrics = {
        **{f"train_{key}": value for key, value in train_metrics.items()},
        **{f"test_{key}": value for key, value in test_metrics.items()},
    }
    return metrics, test_prediction, test_probability, report


metrics, y_pred, y_probability, classification_report_dict = evaluate_new_model(
    model, X_train, X_test, y_train, y_test
)

comparison_results = pd.DataFrame(
    {
        "actual_target": y_test,
        "new_model_prediction": y_pred,
    },
    index=y_test.index,
)
if y_probability is not None:
    comparison_results["new_model_probability"] = y_probability

reference_metrics = None
reference_prediction = None
reference_probability = None

if (
    REFERENCE_PREDICTION_COLUMN
    and REFERENCE_PREDICTION_COLUMN in reference_data.columns
):
    reference_prediction = reference_data.loc[
        test_indices, REFERENCE_PREDICTION_COLUMN
    ]
    valid_reference = reference_prediction.notna()
    reference_truth = y_test.loc[valid_reference]
    reference_prediction_valid = reference_prediction.loc[valid_reference]

    if (
        REFERENCE_PROBABILITY_COLUMN
        and REFERENCE_PROBABILITY_COLUMN in reference_data.columns
    ):
        reference_probability = reference_data.loc[
            test_indices, REFERENCE_PROBABILITY_COLUMN
        ]
        valid_reference &= reference_probability.notna()
        reference_truth = y_test.loc[valid_reference]
        reference_prediction_valid = reference_prediction.loc[valid_reference]
        reference_probability_valid = reference_probability.loc[valid_reference]
    else:
        reference_probability_valid = None

    if TASK_TYPE.lower() == "classification":
        reference_metrics = classification_metrics(
            reference_truth,
            reference_prediction_valid,
            reference_probability_valid,
        )
    else:
        reference_metrics = regression_metrics(
            reference_truth, reference_prediction_valid
        )

    comparison_results["reference_prediction"] = reference_prediction
    if reference_probability is not None:
        comparison_results["reference_probability"] = reference_probability

    if TASK_TYPE.lower() == "classification":
        comparison_results["new_model_correct"] = (
            comparison_results["new_model_prediction"]
            == comparison_results["actual_target"]
        )
        comparison_results["reference_correct"] = (
            comparison_results["reference_prediction"]
            == comparison_results["actual_target"]
        )
        comparison_results["comparison_status"] = np.select(
            [
                comparison_results["new_model_correct"]
                & comparison_results["reference_correct"],
                comparison_results["new_model_correct"]
                & ~comparison_results["reference_correct"],
                ~comparison_results["new_model_correct"]
                & comparison_results["reference_correct"],
            ],
            [
                "both_correct",
                "new_model_only_correct",
                "reference_only_correct",
            ],
            default="both_wrong",
        )

print("New model metrics:")
print(json.dumps(metrics, indent=2, default=str))

if reference_metrics is not None:
    print("\nReference model metrics on the same test rows:")
    print(json.dumps(reference_metrics, indent=2, default=str))
    if not REFERENCE_IS_OUT_OF_SAMPLE:
        print(
            "\nCaution: the reference predictions are retained for diagnostic "
            "comparison, but are not confirmed as out-of-sample predictions."
        )
    if "comparison_status" in comparison_results:
        display(comparison_results["comparison_status"].value_counts().to_frame())


New model metrics:
{
  "train_r2": 0.9460713193809067,
  "train_rmse": 23.950126341699153,
  "train_mae": 11.23995017312727,
  "train_mape": 0.10969521434629088,
  "test_r2": 0.6170616627346055,
  "test_rmse": 61.903948375072275,
  "test_mae": 29.167789882611267,
  "test_mape": 0.29333951677971043
}


## SHAP Explanation

SHAP uses a reproducible sample of the test set when the test set is large. This
controls runtime without changing model training or evaluation.


In [5]:
def select_class_index(model, preferred_class=None):
    if not hasattr(model, "classes_"):
        return 0, None
    classes = list(model.classes_)
    if preferred_class is not None and preferred_class in classes:
        index = classes.index(preferred_class)
    elif len(classes) == 2:
        index = 1
    else:
        index = 0
    return index, classes[index]


def extract_shap_matrix(raw_shap_values, model, task_type):
    if task_type.lower() == "regression":
        if isinstance(raw_shap_values, list):
            return np.asarray(raw_shap_values[0]), None
        return np.asarray(raw_shap_values), None

    class_index, class_label = select_class_index(model, POSITIVE_CLASS)
    if isinstance(raw_shap_values, list):
        return np.asarray(raw_shap_values[class_index]), class_label

    values = np.asarray(raw_shap_values)
    if values.ndim == 3:
        if values.shape[0] == len(getattr(model, "classes_", [])):
            return values[class_index, :, :], class_label
        return values[:, :, class_index], class_label
    return values, class_label


if SHAP_SAMPLE_SIZE and len(X_test) > SHAP_SAMPLE_SIZE:
    X_shap = X_test.sample(n=SHAP_SAMPLE_SIZE, random_state=RANDOM_STATE)
else:
    X_shap = X_test.copy()

explainer = shap.TreeExplainer(model)
try:
    raw_shap_values = explainer.shap_values(
        X_shap,
        check_additivity=False,
        approximate=SHAP_APPROXIMATE,
    )
except TypeError:
    raw_shap_values = explainer.shap_values(
        X_shap,
        approximate=SHAP_APPROXIMATE,
    )

shap_matrix, explained_class = extract_shap_matrix(
    raw_shap_values, model, TASK_TYPE
)

mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
feature_importance = pd.DataFrame(
    {
        "feature": X_shap.columns,
        "mean_abs_shap": mean_abs_shap,
    }
).sort_values("mean_abs_shap", ascending=False)

top_features = feature_importance.head(TOP_K_FEATURES)

print(f"SHAP rows analyzed: {len(X_shap):,}")
if explained_class is not None:
    print(f"SHAP values shown for class: {explained_class}")
display(top_features)


SHAP rows analyzed: 1,000


,feature,mean_abs_shap
25,date_time_sin,24.170251
26,date_time_cos,21.524943
0,lights,7.642585
27,date_week_sin,6.857177
20,Press_mm_hg,4.591877
5,T3,4.543890
6,RH_3,3.807690
15,T8,3.367678
9,T5,3.292538
2,RH_1,3.033839


## Figure Generation

The output package includes SHAP figures, model diagnostics, and, when configured,
a direct new-model versus reference-model comparison.


In [6]:
def write_prompt_templates():
    """Write the three reusable prompt roles used throughout refinement."""
    PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
    templates = {
        "Prompt_1.txt": """I have provided follow-up analytical evidence.

Using this evidence together with the previous interpretation:
1. Explain what the new analysis adds or changes.
2. Update the domain interpretation and practical recommendations.
3. Remove or qualify claims that are not supported by measured evidence.
4. State limitations, uncertainty, and the most important unresolved gap.

Structure the response as:
1. Updated evidence-based insights.
2. Updated recommendations.
3. Remaining limitations and next step.
""",
        "Prompt_2.txt": """Review the current explanation and recommendation.

Identify the most important unresolved analytical gap that can be tested with
the available tabular data. Propose one justified analysis module, the figures
or tables it should generate, and how its result would change the explanation.

Do not invent domain facts, causal effects, or operational recommendations.
Keep the proposed analysis reproducible and suitable for the configured task.
""",
        "Prompt_3.txt": """I have provided the dataset context, held-out model
metrics, SHAP feature importance, a SHAP beeswarm plot, and diagnostic figures.

Based on this evidence:
1. Explain the model behavior and influential features.
2. Distinguish measured evidence from domain assumptions.
3. Provide cautious, evidence-supported recommendations.
4. Identify leakage risks, uncertainty, and analytical gaps.

Structure the response as:
1. Evidence-based insights.
2. Practical recommendations.
3. Limitations and next refinement step.
""",
    }
    for name, text in templates.items():
        (PROMPTS_DIR / name).write_text(text, encoding="utf-8")


def ensure_round_dirs(output_dir, round_id):
    round_dir = Path(output_dir) / f"Round_{round_id}"
    figure_dir = FIGURES_DIR / f"Round_{round_id}"
    prompt_instance_dir = round_dir
    recommendation_dir = RECOMMENDATIONS_DIR
    for directory in [
        round_dir,
        figure_dir,
        PROMPTS_DIR,
        recommendation_dir,
    ]:
        directory.mkdir(parents=True, exist_ok=True)
    write_prompt_templates()
    return (
        round_dir,
        figure_dir,
        prompt_instance_dir,
        recommendation_dir,
    )


def save_figure(fig, figure_dir, name, pdf=None):
    path = figure_dir / f"{name}.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    if pdf is not None:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_model_diagnostics(task_type, y_test, y_pred):
    if task_type.lower() == "regression":
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(y_test, y_pred, alpha=0.75)
        lower = min(np.min(y_test), np.min(y_pred))
        upper = max(np.max(y_test), np.max(y_pred))
        ax.plot([lower, upper], [lower, upper], linestyle="--", color="black")
        ax.set_xlabel("Actual target")
        ax.set_ylabel("Predicted target")
        ax.set_title("New Model: Predicted vs Actual")
        return fig

    fig, ax = plt.subplots(figsize=(7, 6))
    matrix = confusion_matrix(y_test, y_pred)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
    ax.set_title("New Model Confusion Matrix")
    return fig


def plot_reference_comparison(task_type, comparison_results):
    if "reference_prediction" not in comparison_results:
        return None

    valid = comparison_results.dropna(
        subset=["actual_target", "new_model_prediction", "reference_prediction"]
    )
    if valid.empty:
        return None

    if task_type.lower() == "classification":
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        new_matrix = confusion_matrix(
            valid["actual_target"], valid["new_model_prediction"]
        )
        reference_matrix = confusion_matrix(
            valid["actual_target"], valid["reference_prediction"]
        )
        sns.heatmap(new_matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0])
        sns.heatmap(reference_matrix, annot=True, fmt="d", cmap="Oranges", ax=axes[1])
        axes[0].set_title("New Model")
        axes[1].set_title("Reference Model")
        for ax in axes:
            ax.set_xlabel("Predicted class")
            ax.set_ylabel("Actual class")
        fig.suptitle("Model Comparison On The Same Test Rows")
        return fig

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, column, title in [
        (axes[0], "new_model_prediction", "New Model"),
        (axes[1], "reference_prediction", "Reference Model"),
    ]:
        ax.scatter(valid["actual_target"], valid[column], alpha=0.6)
        lower = min(valid["actual_target"].min(), valid[column].min())
        upper = max(valid["actual_target"].max(), valid[column].max())
        ax.plot([lower, upper], [lower, upper], "--", color="black")
        ax.set_title(title)
        ax.set_xlabel("Actual target")
        ax.set_ylabel("Predicted target")
    fig.suptitle("Model Comparison On The Same Test Rows")
    return fig


def plot_group_importance(feature_importance, feature_groups):
    if not feature_groups:
        return None

    rows = []
    for group_name, patterns in feature_groups.items():
        patterns = [str(pattern).lower() for pattern in patterns]
        mask = feature_importance["feature"].str.lower().apply(
            lambda value: any(pattern in value for pattern in patterns)
        )
        rows.append(
            {
                "group": group_name,
                "mean_abs_shap_sum": feature_importance.loc[
                    mask, "mean_abs_shap"
                ].sum(),
            }
        )

    group_df = pd.DataFrame(rows).sort_values(
        "mean_abs_shap_sum", ascending=False
    )
    if group_df.empty or group_df["mean_abs_shap_sum"].sum() == 0:
        return None

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(
        data=group_df,
        y="group",
        x="mean_abs_shap_sum",
        ax=ax,
        color="#4C78A8",
    )
    ax.set_xlabel("Sum of mean absolute SHAP values")
    ax.set_ylabel("Feature group")
    ax.set_title("Feature Group Importance")
    return fig


def generate_figures(
    task_type,
    X_shap,
    y_test,
    y_pred,
    shap_matrix,
    feature_importance,
    feature_groups,
    comparison_results,
    output_dir,
    round_id,
    top_k,
):
    round_dir, figure_dir, prompt_dir, recommendation_dir = ensure_round_dirs(
        output_dir, round_id
    )
    pdf_path = FIGURES_DIR / f"Round_{round_id}.pdf"
    figure_paths = []

    with PdfPages(pdf_path) as pdf:
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.barplot(
            data=feature_importance.head(top_k),
            y="feature",
            x="mean_abs_shap",
            ax=ax,
            color="#4C78A8",
        )
        ax.set_xlabel("Mean absolute SHAP value")
        ax.set_ylabel("Feature")
        ax.set_title(f"Top {top_k} SHAP Feature Importance")
        figure_paths.append(
            save_figure(fig, figure_dir, "01_shap_feature_importance", pdf)
        )

        plt.figure(figsize=(9, 7))
        shap.summary_plot(
            shap_matrix,
            X_shap,
            max_display=top_k,
            show=False,
        )
        fig = plt.gcf()
        fig.suptitle("SHAP Summary Plot", y=1.02)
        figure_paths.append(
            save_figure(fig, figure_dir, "02_shap_summary", pdf)
        )

        fig = plot_model_diagnostics(task_type, y_test, y_pred)
        figure_paths.append(
            save_figure(fig, figure_dir, "03_model_diagnostics", pdf)
        )

        comparison_fig = plot_reference_comparison(
            task_type, comparison_results
        )
        if comparison_fig is not None:
            figure_paths.append(
                save_figure(
                    comparison_fig,
                    figure_dir,
                    "04_reference_model_comparison",
                    pdf,
                )
            )

        top_corr_features = feature_importance.head(
            min(top_k, 12)
        )["feature"].tolist()
        if len(top_corr_features) >= 2:
            fig, ax = plt.subplots(figsize=(10, 8))
            corr = X_shap[top_corr_features].corr()
            sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
            ax.set_title("Correlation Among Top Explained Features")
            figure_paths.append(
                save_figure(fig, figure_dir, "05_top_feature_correlation", pdf)
            )

        group_fig = plot_group_importance(
            feature_importance, feature_groups
        )
        if group_fig is not None:
            figure_paths.append(
                save_figure(
                    group_fig,
                    figure_dir,
                    "06_feature_group_importance",
                    pdf,
                )
            )

    comparison_path = round_dir / "Model_Comparison.csv"
    comparison_results.to_csv(comparison_path, index=True)
    figure_paths.extend([pdf_path, comparison_path])
    return round_dir, figure_dir, prompt_dir, recommendation_dir, figure_paths


round_dir, figure_dir, prompt_dir, recommendation_dir, figure_paths = (
    generate_figures(
        TASK_TYPE,
        X_shap,
        y_test,
        y_pred,
        shap_matrix,
        feature_importance,
        FEATURE_GROUPS,
        comparison_results,
        OUTPUT_DIR,
        ROUND_ID,
        TOP_K_FEATURES,
    )
)

print(f"Saved round outputs to: {round_dir}")
for path in figure_paths:
    print(path)


Saved round outputs to: Evidence\Round_0
Figures\Round_0\01_shap_feature_importance.png
Figures\Round_0\02_shap_summary.png
Figures\Round_0\03_model_diagnostics.png
Figures\Round_0\05_top_feature_correlation.png
Figures\Round_0\06_feature_group_importance.png
Figures\Round_0.pdf
Evidence\Round_0\Model_Comparison.csv


## Agentic XAI Prompt And Recommendation Files

The prompt receives the new-model metrics, reference-model metrics when available,
SHAP evidence, and a clear warning about the evidential limits of the comparison.


In [7]:
def dataset_profile_text(df, X_raw, y, task_type):
    profile = [
        f"Rows: {len(df)}",
        f"Eligible raw feature columns: {X_raw.shape[1]}",
        f"Target column: {TARGET_COLUMN}",
        f"Task type: {task_type}",
        f"Split strategy: {SPLIT_STRATEGY}",
    ]
    if task_type.lower() == "regression":
        profile.extend(
            [
                f"Target mean: {pd.to_numeric(y, errors='coerce').mean():.4f}",
                f"Target standard deviation: "
                f"{pd.to_numeric(y, errors='coerce').std():.4f}",
            ]
        )
    else:
        profile.append("Class distribution:")
        profile.append(y.value_counts().to_string())
    return "\n".join(profile)


def reference_evidence_text(reference_metrics, comparison_results):
    if reference_metrics is None:
        return "No prior-model reference prediction was configured."

    text = [
        "Reference-model metrics on the same selected test rows:",
        json.dumps(reference_metrics, indent=2, default=str),
    ]
    if "comparison_status" in comparison_results:
        text.append("Prediction agreement/error categories:")
        text.append(comparison_results["comparison_status"].value_counts().to_string())
    if not REFERENCE_IS_OUT_OF_SAMPLE:
        text.append(
            "Important limitation: the prior predictions are not confirmed as "
            "out-of-sample. Use them for diagnostic comparison, not as proof that "
            "one model generalizes better."
        )
    return "\n".join(text)


def build_agentic_xai_prompt(
    round_id,
    domain_context,
    task_type,
    metrics,
    reference_metrics,
    top_features,
    figure_paths,
):
    top_feature_lines = [
        f"{idx + 1}. {row.feature}: mean |SHAP| = {row.mean_abs_shap:.6f}"
        for idx, row in enumerate(top_features.itertuples(index=False))
    ]
    figure_lines = [f"- {Path(path).name}" for path in figure_paths]
    metric_lines = [f"- {key}: {value}" for key, value in metrics.items()]

    return f"""You are acting as an Agentic XAI reviewer for a configurable tabular workflow.

Domain context:
{domain_context}

Round:
{round_id}

Dataset and task:
{dataset_profile_text(df, X_raw, y, task_type)}

Model:
Random Forest trained after generic tabular preprocessing.

New-model evaluation metrics:
{chr(10).join(metric_lines)}

Prior-model reference evidence:
{reference_evidence_text(reference_metrics, comparison_results)}

Most influential features according to SHAP:
{chr(10).join(top_feature_lines)}

Generated artifacts:
{chr(10).join(figure_lines)}

Please provide:
1. Domain-grounded interpretation of the new model.
2. Explanation of important features and possible interactions.
3. Comparison with the reference predictions without treating reference fields as inputs.
4. Analysis of cases where only one model was correct.
5. Risks, limitations, leakage concerns, and possible confounders.
6. Concrete recommendations supported by the evidence.
7. Analytical gaps and suggested code/figures for the next refinement round.

Distinguish clearly between measured evidence and domain assumptions.
"""


def write_prompt_and_recommendation(
    prompt_dir, recommendation_dir, round_id, prompt
):
    prompt_path = prompt_dir / "Prompt_Instance.txt"
    prompt_path.write_text(prompt, encoding="utf-8")

    recommendation_path = (
        recommendation_dir / f"Round_{round_id}.md"
    )
    if LLM_PROVIDER.lower() == "manual":
        recommendation_text = f"""# Round {round_id:02d} Recommendations

LLM_PROVIDER is set to manual.

Use the generated prompt, then replace this placeholder with the LLM response.

## Evidence Snapshot

Domain context: {DOMAIN_CONTEXT}

Task type: {TASK_TYPE}

New-model metrics:
{json.dumps(metrics, indent=2, default=str)}

Reference-model metrics:
{json.dumps(reference_metrics, indent=2, default=str)}

Top SHAP features:
{top_features.to_string(index=False)}
"""
    else:
        raise NotImplementedError(
            "Only manual mode is implemented. Add a provider-specific adapter "
            "without changing the core workflow."
        )

    recommendation_path.write_text(recommendation_text, encoding="utf-8")
    return prompt_path, recommendation_path


prompt = build_agentic_xai_prompt(
    ROUND_ID,
    DOMAIN_CONTEXT,
    TASK_TYPE,
    metrics,
    reference_metrics,
    top_features,
    figure_paths,
)

prompt_path, recommendation_path = write_prompt_and_recommendation(
    prompt_dir,
    recommendation_dir,
    ROUND_ID,
    prompt,
)

print(f"Saved prompt: {prompt_path}")
print(f"Saved recommendation draft: {recommendation_path}")


Saved prompt: Evidence\Round_0\Prompt_Instance.txt
Saved recommendation draft: Recommendations\Round_0.md


## Generalized Agentic Refinement Rounds

Rounds 1-4 progressively add reliability, error, feature-behavior, and
robustness evidence. The analyses are dataset-agnostic and use only configured
columns and outputs from Round 0.


In [8]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss


# ============================================================
# Generalized refinement controller and artifact helpers
# ============================================================

REFINEMENT_ROUNDS = {
    1: (
        "generalization_and_calibration"
        if TASK_TYPE.lower() == "classification"
        else "generalization_and_split_sensitivity"
    ),
    2: "error_and_subgroup_audit",
    3: "feature_behavior_and_interactions",
    4: "robustness_and_sensitivity",
}

BOOTSTRAP_ITERATIONS = 200
SUBGROUP_FEATURE_LIMIT = 4
DEPENDENCE_FEATURE_LIMIT = 4
PERMUTATION_FEATURE_LIMIT = 10
PERMUTATION_SAMPLE_SIZE = 2000
PERMUTATION_REPEATS = 2
SHAP_STABILITY_ITERATIONS = 100

workflow_manifest = []


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    return str(value)


def metric_score(y_true, prediction):
    if TASK_TYPE.lower() == "classification":
        return balanced_accuracy_score(y_true, prediction)
    return r2_score(y_true, prediction)


def transformed_to_source(feature_name):
    candidates = sorted(X_raw.columns.astype(str), key=len, reverse=True)
    for source in candidates:
        if feature_name == source or feature_name.startswith(f"{source}_"):
            return source
    return feature_name


def top_source_features(limit):
    ordered = []
    for feature in feature_importance["feature"]:
        source = transformed_to_source(feature)
        if source in X_test_raw.columns and source not in ordered:
            ordered.append(source)
        if len(ordered) >= limit:
            break
    return ordered


def write_refinement_materials(
    round_id,
    analysis_name,
    evidence,
    figure_paths,
    table_paths,
    controller_reason,
    next_analysis,
):
    round_dir, figure_dir, prompt_dir, recommendation_dir = ensure_round_dirs(
        OUTPUT_DIR, round_id
    )
    evidence_path = round_dir / "Evidence.json"
    evidence_path.write_text(
        json.dumps(evidence, indent=2, default=json_default),
        encoding="utf-8",
    )

    prior_recommendation = RECOMMENDATIONS_DIR / f"Round_{round_id - 1}.md"
    artifacts = [*figure_paths, *table_paths, evidence_path]
    artifact_lines = "\n".join(f"- {Path(path).name}" for path in artifacts)

    prompt = f"""You are the self-reflection component of an Agentic XAI workflow.

Domain context:
{DOMAIN_CONTEXT}

Current refinement round:
{round_id}

Current analysis module:
{analysis_name}

Controller decision:
{controller_reason}

Previous recommendation file:
{prior_recommendation}

Measured evidence:
{json.dumps(evidence, indent=2, default=json_default)}

Current artifacts:
{artifact_lines}

Please:
1. Separate measured evidence from domain assumptions.
2. Explain what this round adds beyond the previous round.
3. Identify contradictions, weak evidence, leakage risks, and uncertainty.
4. Update the domain recommendations, keeping only claims supported by evidence.
5. State the most important unresolved analytical gap.
6. Assess whether the proposed next module, {next_analysis}, is justified.

Do not invent economic, operational, causal, or geographic facts that are not
present in the evidence.
"""
    prompt_path = prompt_dir / "Prompt_Instance.txt"
    prompt_path.write_text(prompt, encoding="utf-8")

    recommendation_path = (
        recommendation_dir / f"Round_{round_id}.md"
    )
    recommendation_text = f"""# Round {round_id:02d} Recommendation Draft

## Analysis Module

{analysis_name}

## Controller Decision

{controller_reason}

## Evidence

```json
{json.dumps(evidence, indent=2, default=json_default)}
```

## LLM Self-Reflection

`LLM_PROVIDER` is set to `{LLM_PROVIDER}`. In manual mode, submit the generated
prompt to the same LLM configuration used for every round and replace this
section with its response.

## Proposed Next Analysis

{next_analysis}
"""
    recommendation_path.write_text(recommendation_text, encoding="utf-8")

    decision_path = round_dir / "Controller_Decision.json"
    decision_path.write_text(
        json.dumps(
            {
                "round": round_id,
                "analysis_module": analysis_name,
                "reason": controller_reason,
                "next_analysis": next_analysis,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    record = {
        "round": round_id,
        "analysis_module": analysis_name,
        "controller_reason": controller_reason,
        "next_analysis": next_analysis,
        "figures": [str(path) for path in figure_paths],
        "tables": [str(path) for path in table_paths],
        "evidence": str(evidence_path),
        "prompt": str(prompt_path),
        "recommendation": str(recommendation_path),
    }
    workflow_manifest.append(record)
    return record


def save_round_bundle(round_id, named_figures):
    round_dir, figure_dir, _, _ = ensure_round_dirs(OUTPUT_DIR, round_id)
    pdf_path = FIGURES_DIR / f"Round_{round_id}.pdf"
    saved = []
    with PdfPages(pdf_path) as pdf:
        for name, fig in named_figures:
            saved.append(save_figure(fig, figure_dir, name, pdf))
    saved.append(pdf_path)
    return round_dir, saved


def expected_calibration_error(y_true_binary, probability, bins=10):
    frame = pd.DataFrame(
        {"truth": np.asarray(y_true_binary), "probability": probability}
    )
    frame["bin"] = pd.cut(
        frame["probability"],
        bins=np.linspace(0, 1, bins + 1),
        include_lowest=True,
    )
    grouped = frame.groupby("bin", observed=False).agg(
        count=("truth", "size"),
        observed_rate=("truth", "mean"),
        mean_probability=("probability", "mean"),
    )
    grouped = grouped.dropna()
    if grouped.empty:
        return np.nan
    weights = grouped["count"] / grouped["count"].sum()
    return float(
        (
            weights
            * (grouped["observed_rate"] - grouped["mean_probability"]).abs()
        ).sum()
    )


### Round 1 - Generalization And Split Sensitivity

Quantify the train-test gap and uncertainty. For dated regression data, challenge
the conventional random-row baseline using grouped-date and chronological future
holdouts. This tests whether apparently strong performance depends on neighboring
observations or a stationary time period.


In [9]:
# Round 1: Generalization, uncertainty, and calibration

rng = np.random.default_rng(RANDOM_STATE)

if TASK_TYPE.lower() == "classification":
    metric_names = ["accuracy", "balanced_accuracy", "precision", "recall", "f1"]
else:
    metric_names = ["r2", "rmse", "mae"]

metric_rows = []
for name in metric_names:
    metric_rows.append(
        {
            "metric": name,
            "train": metrics.get(f"train_{name}"),
            "test": metrics.get(f"test_{name}"),
        }
    )
train_test_metrics = pd.DataFrame(metric_rows)

bootstrap_rows = []
n_test = len(y_test)
for iteration in range(BOOTSTRAP_ITERATIONS):
    positions = rng.integers(0, n_test, size=n_test)
    truth = y_test.to_numpy()[positions]
    prediction = np.asarray(y_pred)[positions]
    probability = (
        np.asarray(y_probability)[positions]
        if y_probability is not None
        else None
    )
    try:
        if TASK_TYPE.lower() == "classification":
            values = classification_metrics(truth, prediction, probability)
        else:
            values = regression_metrics(truth, prediction)
        values["iteration"] = iteration
        bootstrap_rows.append(values)
    except ValueError:
        continue

bootstrap_metrics = pd.DataFrame(bootstrap_rows)
bootstrap_intervals = []
for column in bootstrap_metrics.columns:
    if column == "iteration":
        continue
    bootstrap_intervals.append(
        {
            "metric": column,
            "mean": bootstrap_metrics[column].mean(),
            "ci_2_5": bootstrap_metrics[column].quantile(0.025),
            "ci_97_5": bootstrap_metrics[column].quantile(0.975),
        }
    )
bootstrap_intervals = pd.DataFrame(bootstrap_intervals)


def evaluate_regression_split(split_name, split_train_indices, split_test_indices):
    split_X_train_raw = X_raw.loc[split_train_indices]
    split_X_test_raw = X_raw.loc[split_test_indices]
    split_y_train = y.loc[split_train_indices]
    split_y_test = y.loc[split_test_indices]

    split_preprocessor, _, _ = make_preprocessor(split_X_train_raw)
    split_X_train_array = split_preprocessor.fit_transform(split_X_train_raw)
    split_X_test_array = split_preprocessor.transform(split_X_test_raw)
    split_feature_names = get_feature_names(split_preprocessor)
    split_X_train = pd.DataFrame(
        split_X_train_array,
        columns=split_feature_names,
        index=split_X_train_raw.index,
    )
    split_X_test = pd.DataFrame(
        split_X_test_array,
        columns=split_feature_names,
        index=split_X_test_raw.index,
    )

    split_model = make_model(TASK_TYPE, MODEL_PARAMS)
    split_model.fit(split_X_train, split_y_train)
    split_train_prediction = split_model.predict(split_X_train)
    split_test_prediction = split_model.predict(split_X_test)
    split_train_metrics = regression_metrics(
        split_y_train,
        split_train_prediction,
    )
    split_test_metrics = regression_metrics(
        split_y_test,
        split_test_prediction,
    )
    row = {
        "split": split_name,
        "train_rows": len(split_train_indices),
        "test_rows": len(split_test_indices),
        **{
            f"train_{key}": value
            for key, value in split_train_metrics.items()
        },
        **{
            f"test_{key}": value
            for key, value in split_test_metrics.items()
        },
    }
    return row, split_y_test, split_test_prediction


split_sensitivity_rows = [
    {
        "split": SPLIT_STRATEGY,
        "train_rows": len(train_indices),
        "test_rows": len(test_indices),
        **metrics,
    }
]
temporal_stress_predictions = pd.DataFrame()

if (
    TASK_TYPE.lower() == "regression"
    and RUN_DATE_SPLIT_STRESS_TESTS
    and TIME_COLUMN in df.columns
):
    split_times = pd.to_datetime(df[TIME_COLUMN], errors="coerce")
    if split_times.isna().any():
        raise ValueError("Date split stress tests require valid timestamps.")

    date_groups = split_times.dt.to_period(DATE_GROUP_FREQUENCY).astype(str)
    date_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    grouped_train_positions, grouped_test_positions = next(
        date_splitter.split(df, y, date_groups)
    )
    grouped_train_indices = df.index[grouped_train_positions]
    grouped_test_indices = df.index[grouped_test_positions]
    grouped_row, _, _ = evaluate_regression_split(
        "grouped_dates",
        grouped_train_indices,
        grouped_test_indices,
    )
    split_sensitivity_rows.append(grouped_row)

    ordered_indices = split_times.sort_values().index.to_numpy()
    temporal_cut = int(len(ordered_indices) * (1 - TEST_SIZE))
    temporal_train_indices = ordered_indices[:temporal_cut]
    temporal_test_indices = ordered_indices[temporal_cut:]
    temporal_row, temporal_y_test, temporal_prediction = (
        evaluate_regression_split(
            "temporal_future",
            temporal_train_indices,
            temporal_test_indices,
        )
    )
    split_sensitivity_rows.append(temporal_row)
    temporal_stress_predictions = pd.DataFrame(
        {
            "time": split_times.loc[temporal_test_indices],
            "actual": temporal_y_test,
            "prediction": temporal_prediction,
            "residual": temporal_y_test.to_numpy() - temporal_prediction,
        },
        index=temporal_test_indices,
    ).sort_values("time")

split_sensitivity_metrics = pd.DataFrame(split_sensitivity_rows)

figures_round_1 = []
plot_metrics = train_test_metrics.melt(
    id_vars="metric",
    value_vars=["train", "test"],
    var_name="split",
    value_name="value",
)
if TASK_TYPE.lower() == "regression":
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.barplot(
        data=plot_metrics[plot_metrics["metric"] == "r2"],
        x="metric",
        y="value",
        hue="split",
        ax=axes[0],
    )
    axes[0].set_title("Train-Test R2")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("R2")
    sns.barplot(
        data=plot_metrics[plot_metrics["metric"].isin(["rmse", "mae"])],
        x="metric",
        y="value",
        hue="split",
        ax=axes[1],
    )
    axes[1].set_title("Train-Test Error")
    axes[1].set_xlabel("Metric")
    axes[1].set_ylabel("Target units")
    fig.tight_layout()
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=plot_metrics, x="metric", y="value", hue="split", ax=ax)
    ax.set_title("Train-Test Generalization Comparison")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Metric value")
    ax.tick_params(axis="x", rotation=25)
figures_round_1.append(("01_train_test_generalization", fig))

if len(split_sensitivity_metrics) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.barplot(
        data=split_sensitivity_metrics,
        x="split",
        y="test_r2",
        color="#4C78A8",
        ax=axes[0],
    )
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title("R2 Sensitivity to Evaluation Split")
    axes[0].set_xlabel("Evaluation design")
    axes[0].set_ylabel("Test R2")
    axes[0].tick_params(axis="x", rotation=20)
    split_errors = split_sensitivity_metrics.melt(
        id_vars="split",
        value_vars=["test_rmse", "test_mae"],
        var_name="metric",
        value_name="value",
    )
    sns.barplot(
        data=split_errors,
        x="split",
        y="value",
        hue="metric",
        ax=axes[1],
    )
    axes[1].set_title("Error Sensitivity to Evaluation Split")
    axes[1].set_xlabel("Evaluation design")
    axes[1].set_ylabel("Target units")
    axes[1].tick_params(axis="x", rotation=20)
    fig.tight_layout()
    figures_round_1.append(("02_split_sensitivity", fig))

if not temporal_stress_predictions.empty:
    rolling_window = max(
        12,
        min(144, len(temporal_stress_predictions) // 20),
    )
    temporal_stress_predictions["rolling_absolute_error"] = (
        temporal_stress_predictions["residual"].abs().rolling(
            rolling_window,
            min_periods=max(3, rolling_window // 4),
        ).mean()
    )
    fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
    axes[0].plot(
        temporal_stress_predictions["time"],
        temporal_stress_predictions["actual"],
        label="Actual",
        linewidth=1,
        alpha=0.75,
    )
    axes[0].plot(
        temporal_stress_predictions["time"],
        temporal_stress_predictions["prediction"],
        label="Predicted",
        linewidth=1,
        alpha=0.75,
    )
    axes[0].set_ylabel("Target")
    axes[0].set_title("Future-period Stress Test: Actual vs Predicted")
    axes[0].legend()
    axes[1].plot(
        temporal_stress_predictions["time"],
        temporal_stress_predictions["rolling_absolute_error"],
        color="#E15759",
    )
    axes[1].set_ylabel("Rolling MAE")
    axes[1].set_xlabel("Time")
    axes[1].set_title(
        f"Future-period Rolling Error (window={rolling_window} observations)"
    )
    fig.tight_layout()
    figures_round_1.append(("03_future_period_stress_test", fig))

calibration_evidence = {}
if (
    TASK_TYPE.lower() == "classification"
    and y_probability is not None
    and pd.Series(y_test).nunique() == 2
):
    binary_truth = (y_test.to_numpy() == POSITIVE_CLASS).astype(int)
    observed_rate, mean_probability = calibration_curve(
        binary_truth,
        y_probability,
        n_bins=10,
        strategy="quantile",
    )
    brier = float(brier_score_loss(binary_truth, y_probability))
    ece = expected_calibration_error(binary_truth, y_probability)
    calibration_evidence = {
        "brier_score": brier,
        "expected_calibration_error": ece,
    }

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot([0, 1], [0, 1], "--", color="black", label="Ideal")
    axes[0].plot(
        mean_probability,
        observed_rate,
        marker="o",
        color="#D1495B",
        label="New model",
    )
    axes[0].set_xlabel("Mean predicted probability")
    axes[0].set_ylabel("Observed positive rate")
    axes[0].set_title("Probability Calibration")
    axes[0].legend()
    axes[1].hist(y_probability, bins=20, color="#4C78A8", edgecolor="white")
    axes[1].set_xlabel("Predicted positive-class probability")
    axes[1].set_ylabel("Test observations")
    axes[1].set_title("Prediction Probability Distribution")
    figures_round_1.append(("02_probability_calibration", fig))

    confidence = np.abs(np.asarray(y_probability) - 0.5) * 2
    confidence_frame = pd.DataFrame(
        {
            "confidence": confidence,
            "correct": np.asarray(y_pred) == y_test.to_numpy(),
        }
    )
    confidence_frame["confidence_bin"] = pd.cut(
        confidence_frame["confidence"],
        bins=np.linspace(0, 1, 6),
        include_lowest=True,
    )
    confidence_summary = (
        confidence_frame.groupby("confidence_bin", observed=False)
        .agg(
            observations=("correct", "size"),
            accuracy=("correct", "mean"),
            mean_confidence=("confidence", "mean"),
        )
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(
        data=confidence_summary,
        x="confidence_bin",
        y="accuracy",
        color="#59A14F",
        ax=ax,
    )
    ax.set_ylim(0, 1)
    ax.set_xlabel("Model confidence")
    ax.set_ylabel("Observed accuracy")
    ax.set_title("Accuracy by Prediction Confidence")
    ax.tick_params(axis="x", rotation=25)
    figures_round_1.append(("03_confidence_reliability", fig))
else:
    residual = y_test.to_numpy() - np.asarray(y_pred)
    residual_frame = pd.DataFrame(
        {"prediction": y_pred, "residual": residual}
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(y_pred, residual, alpha=0.45)
    axes[0].axhline(0, linestyle="--", color="black")
    axes[0].set_xlabel("Prediction")
    axes[0].set_ylabel("Residual")
    axes[0].set_title("Residual Pattern")
    axes[1].hist(residual, bins=25, color="#4C78A8", edgecolor="white")
    axes[1].set_xlabel("Residual")
    axes[1].set_ylabel("Test observations")
    axes[1].set_title("Residual Distribution")
    figures_round_1.append(("04_residual_reliability", fig))
    confidence_summary = residual_frame.describe().reset_index()

    if SPLIT_STRATEGY.lower() == "temporal" and TIME_COLUMN in df.columns:
        temporal_residuals = pd.DataFrame(
            {
                "time": pd.to_datetime(df.loc[test_indices, TIME_COLUMN]),
                "actual": y_test,
                "prediction": y_pred,
                "residual": residual,
            },
            index=y_test.index,
        ).sort_values("time")
        rolling_window = max(12, min(144, len(temporal_residuals) // 20))
        temporal_residuals["rolling_absolute_error"] = (
            temporal_residuals["residual"].abs().rolling(
                rolling_window,
                min_periods=max(3, rolling_window // 4),
            ).mean()
        )

        fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
        axes[0].plot(
            temporal_residuals["time"],
            temporal_residuals["actual"],
            label="Actual",
            linewidth=1,
            alpha=0.75,
        )
        axes[0].plot(
            temporal_residuals["time"],
            temporal_residuals["prediction"],
            label="Predicted",
            linewidth=1,
            alpha=0.75,
        )
        axes[0].set_ylabel("Target")
        axes[0].set_title("Held-out Future Period: Actual vs Predicted")
        axes[0].legend()
        axes[1].plot(
            temporal_residuals["time"],
            temporal_residuals["rolling_absolute_error"],
            color="#E15759",
        )
        axes[1].set_ylabel("Rolling MAE")
        axes[1].set_xlabel("Time")
        axes[1].set_title(
            f"Rolling Absolute Error (window={rolling_window} observations)"
        )
        fig.tight_layout()
        figures_round_1.append(("05_temporal_residual_diagnostics", fig))

round_1_dir, round_1_figures = save_round_bundle(1, figures_round_1)
round_1_tables = []
for name, table in [
    ("train_test_metrics.csv", train_test_metrics),
    ("bootstrap_metric_intervals.csv", bootstrap_intervals),
    ("confidence_or_residual_summary.csv", confidence_summary),
    ("split_sensitivity_metrics.csv", split_sensitivity_metrics),
    ("temporal_stress_predictions.csv", temporal_stress_predictions),
]:
    path = round_1_dir / name
    table.to_csv(path, index=False)
    round_1_tables.append(path)

if TASK_TYPE.lower() == "classification":
    generalization_gap = (
        metrics.get("train_balanced_accuracy", np.nan)
        - metrics.get("test_balanced_accuracy", np.nan)
    )
    gap_label = "balanced-accuracy train-test gap"
else:
    generalization_gap = (
        metrics.get("train_r2", np.nan) - metrics.get("test_r2", np.nan)
    )
    gap_label = "R2 train-test gap"

severity = "large" if generalization_gap > 0.10 else "moderate or small"
if len(split_sensitivity_metrics) > 1:
    split_r2 = split_sensitivity_metrics.set_index("split")["test_r2"]
    random_r2 = split_r2.get(SPLIT_STRATEGY, np.nan)
    grouped_r2 = split_r2.get("grouped_dates", np.nan)
    temporal_r2 = split_r2.get("temporal_future", np.nan)
    controller_reason_1 = (
        f"The conventional {SPLIT_STRATEGY} test R2 is {random_r2:.4f}, "
        f"but grouped-date R2 is {grouped_r2:.4f} and future-period R2 is "
        f"{temporal_r2:.4f}. This large split sensitivity indicates that "
        "neighboring observations and temporal distribution shift materially "
        "affect the apparent model quality. The next round must locate where "
        "errors concentrate."
    )
else:
    controller_reason_1 = (
        f"The {gap_label} is {generalization_gap:.4f}, classified as {severity}. "
        "The next round must locate where errors concentrate rather than adding "
        "more global importance plots."
    )
round_1_evidence = {
    "train_test_metrics": train_test_metrics.to_dict(orient="records"),
    "bootstrap_intervals": bootstrap_intervals.to_dict(orient="records"),
    "generalization_gap": generalization_gap,
    "calibration": calibration_evidence,
    "split_sensitivity": split_sensitivity_metrics.to_dict(orient="records"),
}
round_1_record = write_refinement_materials(
    1,
    REFINEMENT_ROUNDS[1],
    round_1_evidence,
    round_1_figures,
    round_1_tables,
    controller_reason_1,
    REFINEMENT_ROUNDS[2],
)

display(train_test_metrics)
display(bootstrap_intervals)
print(controller_reason_1)
print(f"Saved Round 1 to: {round_1_dir}")


,metric,train,test
0,r2,0.946071,0.617062
1,rmse,23.950126,61.903948
2,mae,11.239950,29.167790


,metric,mean,ci_2_5,ci_97_5
0,r2,0.614278,0.581107,0.644136
1,rmse,62.083707,57.572754,67.445876
2,mae,29.192537,27.360399,31.086441
3,mape,0.293597,0.281395,0.306422


The conventional random test R2 is 0.6171, but grouped-date R2 is -0.0527 and future-period R2 is -3.6599. This large split sensitivity indicates that neighboring observations and temporal distribution shift materially affect the apparent model quality. The next round must locate where errors concentrate.
Saved Round 1 to: Evidence\Round_1


### Round 2 - Error And Subgroup Audit

Locate concentrated errors across influential raw features, reference-model
disagreements, and configured spatial coordinates.


In [10]:
# Round 2: Error, disagreement, subgroup, and optional spatial audit

audit_features = top_source_features(SUBGROUP_FEATURE_LIMIT)
subgroup_rows = []
minimum_group_size = max(25, int(len(y_test) * 0.005))

for feature in audit_features:
    values = X_test_raw[feature]
    if pd.api.types.is_numeric_dtype(values) and values.nunique(dropna=True) > 8:
        try:
            groups = pd.qcut(values, q=4, duplicates="drop")
        except ValueError:
            groups = values.astype(str)
    else:
        groups = values.fillna("Missing").astype(str)

    frame = pd.DataFrame(
        {
            "group": groups.astype(str),
            "truth": y_test,
            "prediction": y_pred,
        },
        index=y_test.index,
    )
    for group_name, group in frame.groupby("group", dropna=False):
        if len(group) < minimum_group_size:
            continue
        if TASK_TYPE.lower() == "classification":
            row = {
                "feature": feature,
                "group": group_name,
                "count": len(group),
                "group_order": pd.to_numeric(
                    values.loc[group.index], errors="coerce"
                ).median(),
                "accuracy": accuracy_score(
                    group["truth"], group["prediction"]
                ),
                "error_rate": 1
                - accuracy_score(group["truth"], group["prediction"]),
            }
            if group["truth"].nunique() > 1:
                row["balanced_accuracy"] = balanced_accuracy_score(
                    group["truth"], group["prediction"]
                )
            else:
                row["balanced_accuracy"] = np.nan
            positive_mask = group["truth"] == POSITIVE_CLASS
            row["positive_recall"] = (
                recall_score(
                    group["truth"],
                    group["prediction"],
                    pos_label=POSITIVE_CLASS,
                    zero_division=0,
                )
                if positive_mask.any()
                else np.nan
            )
        else:
            residual = group["truth"] - group["prediction"]
            row = {
                "feature": feature,
                "group": group_name,
                "count": len(group),
                "group_order": pd.to_numeric(
                    values.loc[group.index], errors="coerce"
                ).median(),
                "mae": mean_absolute_error(
                    group["truth"], group["prediction"]
                ),
                "rmse": float(
                    np.sqrt(
                        mean_squared_error(
                            group["truth"], group["prediction"]
                        )
                    )
                ),
                "mean_residual": residual.mean(),
            }
        subgroup_rows.append(row)

subgroup_audit = pd.DataFrame(subgroup_rows)
figures_round_2 = []

if not subgroup_audit.empty:
    performance_column = (
        "error_rate" if TASK_TYPE.lower() == "classification" else "mae"
    )
    features_for_plot = subgroup_audit["feature"].unique().tolist()
    fig, axes = plt.subplots(
        len(features_for_plot),
        1,
        figsize=(10, max(4, 3.5 * len(features_for_plot))),
        squeeze=False,
    )
    for ax, feature in zip(axes[:, 0], features_for_plot):
        subset = subgroup_audit[subgroup_audit["feature"] == feature].copy()
        if subset["group_order"].notna().any():
            subset = subset.sort_values("group_order")
        sns.barplot(
            data=subset,
            x="group",
            y=performance_column,
            color="#E15759",
            ax=ax,
        )
        ax.set_title(f"{performance_column.replace('_', ' ').title()} by {feature}")
        ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    figures_round_2.append(("01_subgroup_performance", fig))

if "comparison_status" in comparison_results.columns:
    status_counts = (
        comparison_results["comparison_status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="count")
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(
        data=status_counts,
        x="status",
        y="count",
        hue="status",
        legend=False,
        ax=ax,
    )
    ax.set_title("New Model and Reference Prediction Agreement")
    ax.set_xlabel("Comparison status")
    ax.set_ylabel("Test observations")
    ax.tick_params(axis="x", rotation=20)
    figures_round_2.append(("02_reference_disagreement", fig))
else:
    status_counts = pd.DataFrame()

coordinate_columns = [
    column
    for column in [*GROUP_COLUMNS, *ID_COLUMNS]
    if column in df.columns and pd.api.types.is_numeric_dtype(df[column])
]
coordinate_columns = list(dict.fromkeys(coordinate_columns))
if len(coordinate_columns) >= 2:
    x_column, y_column = coordinate_columns[:2]
    spatial_frame = df.loc[
        test_indices, [x_column, y_column]
    ].copy()
    if TASK_TYPE.lower() == "classification":
        spatial_frame["diagnostic"] = np.where(
            np.asarray(y_pred) == y_test.to_numpy(), "Correct", "Error"
        )
    else:
        spatial_frame["diagnostic"] = np.abs(
            y_test.to_numpy() - np.asarray(y_pred)
        )
    if len(spatial_frame) > 10000:
        spatial_frame = spatial_frame.sample(
            10000, random_state=RANDOM_STATE
        )

    fig, ax = plt.subplots(figsize=(8, 7))
    if TASK_TYPE.lower() == "classification":
        sns.scatterplot(
            data=spatial_frame,
            x=x_column,
            y=y_column,
            hue="diagnostic",
            hue_order=["Correct", "Error"],
            palette={"Correct": "#59A14F", "Error": "#E15759"},
            alpha=0.55,
            s=18,
            ax=ax,
        )
    else:
        scatter = ax.scatter(
            spatial_frame[x_column],
            spatial_frame[y_column],
            c=spatial_frame["diagnostic"],
            cmap="magma",
            alpha=0.6,
            s=18,
        )
        fig.colorbar(scatter, ax=ax, label="Absolute error")
    ax.set_title("Spatial Distribution of Test Errors")
    figures_round_2.append(("03_spatial_error_audit", fig))

round_2_dir, round_2_figures = save_round_bundle(2, figures_round_2)
round_2_tables = []
for name, table in [
    ("subgroup_audit.csv", subgroup_audit),
    ("reference_disagreement_counts.csv", status_counts),
]:
    path = round_2_dir / name
    table.to_csv(path, index=False)
    round_2_tables.append(path)

if not subgroup_audit.empty:
    performance_column = (
        "error_rate" if TASK_TYPE.lower() == "classification" else "mae"
    )
    worst = subgroup_audit.sort_values(
        performance_column, ascending=False
    ).iloc[0]
    worst_group = {
        "feature": worst["feature"],
        "group": worst["group"],
        "count": int(worst["count"]),
        performance_column: float(worst[performance_column]),
    }
else:
    worst_group = None

controller_reason_2 = (
    "The global metrics hide subgroup-dependent behavior. "
    f"The worst observed subgroup is {worst_group}. The next round examines "
    "how influential features change predictions and whether apparent effects "
    "are conditional on other features."
)
round_2_evidence = {
    "audited_source_features": audit_features,
    "minimum_group_size": minimum_group_size,
    "worst_subgroup": worst_group,
    "reference_disagreement_counts": (
        status_counts.to_dict(orient="records")
        if not status_counts.empty
        else []
    ),
    "spatial_columns_used": coordinate_columns[:2],
}
round_2_record = write_refinement_materials(
    2,
    REFINEMENT_ROUNDS[2],
    round_2_evidence,
    round_2_figures,
    round_2_tables,
    controller_reason_2,
    REFINEMENT_ROUNDS[3],
)

display(subgroup_audit.head(20))
print(controller_reason_2)
print(f"Saved Round 2 to: {round_2_dir}")


,feature,group,count,group_order,mae,rmse,mean_residual
0,date_time_sin,"(-0.737, 0.0]",997,-0.382683,23.146506,46.269178,-2.704982
1,date_time_sin,"(-1.001, -0.737]",992,-0.923880,43.981384,79.832398,-0.173320
2,date_time_sin,"(0.0, 0.676]",978,0.382683,27.821575,66.628046,-0.592604
3,date_time_sin,"(0.676, 1.0]",980,0.923880,21.642007,48.728964,-0.473503
4,date_time_cos,"(-0.0436, 0.707]",1002,0.382683,28.387725,62.090938,0.364837
5,date_time_cos,"(-0.737, -0.0436]",986,-0.422618,37.161089,69.484818,-1.961697
6,date_time_cos,"(-1.001, -0.737]",992,-0.923880,40.498589,77.419938,-1.879839
7,date_time_cos,"(0.707, 1.0]",967,0.923880,10.201999,23.975533,-0.495277
8,lights,0,3047,0.000000,23.989257,52.312969,-2.626584
9,lights,10,446,10.000000,45.645590,78.917870,-3.302541


The global metrics hide subgroup-dependent behavior. The worst observed subgroup is {'feature': 'lights', 'group': '30', 'count': 104, 'mae': 50.42532051282051}. The next round examines how influential features change predictions and whether apparent effects are conditional on other features.
Saved Round 2 to: Evidence\Round_2


### Round 3 - Feature Behavior And Interaction Screening

Move beyond global importance by examining SHAP direction, binned effects, and
contribution co-movement. Co-movement is explicitly treated as a screening
proxy, not a causal interaction.


In [11]:
# Round 3: Feature behavior and interaction evidence

effect_features = [
    feature
    for feature in feature_importance["feature"].tolist()
    if feature in X_shap.columns
][:DEPENDENCE_FEATURE_LIMIT]
shap_frame = pd.DataFrame(
    shap_matrix,
    columns=X_shap.columns,
    index=X_shap.index,
)

effect_rows = []
effect_summary_rows = []
for feature in effect_features:
    values = X_shap[feature]
    contributions = shap_frame[feature]
    unique_values = values.nunique(dropna=True)
    if unique_values > 8:
        try:
            bins = pd.qcut(values, q=8, duplicates="drop")
        except ValueError:
            bins = pd.cut(values, bins=8, duplicates="drop")
    else:
        bins = values.astype(str)

    local = pd.DataFrame(
        {
            "feature_value": values,
            "shap_value": contributions,
            "bin": bins.astype(str),
        }
    )
    grouped = (
        local.groupby("bin", observed=False)
        .agg(
            count=("shap_value", "size"),
            median_feature_value=("feature_value", "median"),
            mean_shap=("shap_value", "mean"),
            mean_abs_shap=("shap_value", lambda series: series.abs().mean()),
        )
        .reset_index()
    )
    grouped = grouped.sort_values("median_feature_value").reset_index(drop=True)
    grouped.insert(0, "feature", feature)
    effect_rows.extend(grouped.to_dict(orient="records"))
    effect_summary_rows.append(
        {
            "feature": feature,
            "spearman_value_shap": values.corr(
                contributions, method="spearman"
            ),
            "mean_abs_shap": contributions.abs().mean(),
        }
    )

feature_effects = pd.DataFrame(effect_rows)
feature_effect_summary = pd.DataFrame(effect_summary_rows)

figures_round_3 = []
if effect_features:
    rows = int(np.ceil(len(effect_features) / 2))
    fig, axes = plt.subplots(
        rows,
        2,
        figsize=(13, max(5, rows * 4.5)),
        squeeze=False,
    )
    color_feature = effect_features[1] if len(effect_features) > 1 else None
    for index, feature in enumerate(effect_features):
        ax = axes.flat[index]
        color_values = (
            X_shap[color_feature]
            if color_feature and color_feature != feature
            else None
        )
        scatter = ax.scatter(
            X_shap[feature],
            shap_frame[feature],
            c=color_values if color_values is not None else "#4C78A8",
            cmap="viridis" if color_values is not None else None,
            alpha=0.55,
            s=18,
        )
        ax.axhline(0, linestyle="--", color="black", linewidth=1)
        ax.set_xlabel(feature)
        ax.set_ylabel("SHAP contribution")
        ax.set_title(f"Effect of {feature}")
        if color_values is not None:
            fig.colorbar(scatter, ax=ax, label=color_feature)
    for index in range(len(effect_features), len(axes.flat)):
        axes.flat[index].axis("off")
    fig.tight_layout()
    figures_round_3.append(("01_shap_dependence", fig))

if not feature_effects.empty:
    features_for_plot = feature_effects["feature"].unique().tolist()
    fig, axes = plt.subplots(
        len(features_for_plot),
        1,
        figsize=(10, max(4, 3.5 * len(features_for_plot))),
        squeeze=False,
    )
    for ax, feature in zip(axes[:, 0], features_for_plot):
        subset = feature_effects[feature_effects["feature"] == feature]
        ax.plot(
            np.arange(len(subset)),
            subset["mean_shap"],
            marker="o",
            color="#F28E2B",
        )
        ax.axhline(0, linestyle="--", color="black", linewidth=1)
        ax.set_xticks(np.arange(len(subset)))
        ax.set_xticklabels(subset["bin"], rotation=25, ha="right")
        ax.set_ylabel("Mean SHAP")
        ax.set_title(f"Binned Effect Profile: {feature}")
    fig.tight_layout()
    figures_round_3.append(("02_binned_feature_effects", fig))

interaction_proxy = pd.DataFrame()
if len(effect_features) >= 2:
    contribution_corr = shap_frame[effect_features].corr(method="spearman")
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        contribution_corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        ax=ax,
    )
    ax.set_title(
        "SHAP Contribution Co-movement\n"
        "(screening proxy, not formal SHAP interaction values)"
    )
    figures_round_3.append(("03_shap_comovement_proxy", fig))

    pairs = []
    for i, left in enumerate(effect_features):
        for right in effect_features[i + 1 :]:
            pairs.append(
                {
                    "feature_1": left,
                    "feature_2": right,
                    "spearman_shap_comovement": contribution_corr.loc[
                        left, right
                    ],
                }
            )
    interaction_proxy = pd.DataFrame(pairs).sort_values(
        "spearman_shap_comovement",
        key=lambda series: series.abs(),
        ascending=False,
    )

round_3_dir, round_3_figures = save_round_bundle(3, figures_round_3)
round_3_tables = []
for name, table in [
    ("feature_effects.csv", feature_effects),
    ("feature_effect_summary.csv", feature_effect_summary),
    ("interaction_screening_proxy.csv", interaction_proxy),
]:
    path = round_3_dir / name
    table.to_csv(path, index=False)
    round_3_tables.append(path)

strongest_effect = (
    feature_effect_summary.sort_values(
        "mean_abs_shap", ascending=False
    ).iloc[0].to_dict()
    if not feature_effect_summary.empty
    else None
)
strongest_pair = (
    interaction_proxy.iloc[0].to_dict()
    if not interaction_proxy.empty
    else None
)
controller_reason_3 = (
    "Global importance alone does not establish direction, thresholds, or "
    "conditional behavior. This round characterized binned SHAP effects and "
    f"identified the strongest contribution co-movement pair as {strongest_pair}. "
    "The final round tests whether these explanations remain stable under "
    "permutation, perturbation, and SHAP resampling."
)
round_3_evidence = {
    "features_analyzed": effect_features,
    "strongest_feature_effect": strongest_effect,
    "strongest_comovement_pair": strongest_pair,
    "interaction_warning": (
        "Contribution co-movement is a screening proxy and must not be "
        "reported as a causal or formal interaction estimate."
    ),
}
round_3_record = write_refinement_materials(
    3,
    REFINEMENT_ROUNDS[3],
    round_3_evidence,
    round_3_figures,
    round_3_tables,
    controller_reason_3,
    REFINEMENT_ROUNDS[4],
)

display(feature_effect_summary)
display(interaction_proxy.head(10))
print(controller_reason_3)
print(f"Saved Round 3 to: {round_3_dir}")


,feature,spearman_value_shap,mean_abs_shap
0,date_time_sin,-0.805649,24.170251
1,date_time_cos,-0.901993,21.524943
2,lights,0.751972,7.642585
3,date_week_sin,-0.282966,6.857177


,feature_1,feature_2,spearman_shap_comovement
0,date_time_sin,date_time_cos,0.431516
3,date_time_cos,lights,-0.232103
5,lights,date_week_sin,0.210295
4,date_time_cos,date_week_sin,-0.204874
1,date_time_sin,lights,-0.193816
2,date_time_sin,date_week_sin,-0.105448


Global importance alone does not establish direction, thresholds, or conditional behavior. This round characterized binned SHAP effects and identified the strongest contribution co-movement pair as {'feature_1': 'date_time_sin', 'feature_2': 'date_time_cos', 'spearman_shap_comovement': 0.4315161715161715}. The final round tests whether these explanations remain stable under permutation, perturbation, and SHAP resampling.
Saved Round 3 to: Evidence\Round_3


### Round 4 - Robustness And Conditional Stop

Test held-out permutation sensitivity, local perturbation response, and SHAP
rank stability. The workflow stops unless self-reflection identifies a
specific evidence-backed gap.


In [12]:
# Round 4: Robustness, sensitivity, and final evidence synthesis

if len(X_test) > PERMUTATION_SAMPLE_SIZE:
    robustness_indices = X_test.sample(
        PERMUTATION_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
    ).index
else:
    robustness_indices = X_test.index

X_robust = X_test.loc[robustness_indices].copy()
y_robust = y_test.loc[robustness_indices]
baseline_prediction = model.predict(X_robust)
baseline_score = metric_score(y_robust, baseline_prediction)

rng = np.random.default_rng(RANDOM_STATE)
permutation_rows = []
permutation_features = [
    feature
    for feature in feature_importance["feature"].tolist()
    if feature in X_robust.columns
][:PERMUTATION_FEATURE_LIMIT]

for feature in permutation_features:
    repeat_scores = []
    original = X_robust[feature].to_numpy(copy=True)
    for repeat in range(PERMUTATION_REPEATS):
        permuted = X_robust.copy()
        permuted[feature] = rng.permutation(original)
        repeat_prediction = model.predict(permuted)
        repeat_scores.append(metric_score(y_robust, repeat_prediction))
    permutation_rows.append(
        {
            "feature": feature,
            "baseline_score": baseline_score,
            "permuted_score_mean": np.mean(repeat_scores),
            "score_drop": baseline_score - np.mean(repeat_scores),
            "score_drop_std": np.std(repeat_scores),
        }
    )

permutation_sensitivity = pd.DataFrame(permutation_rows).sort_values(
    "score_drop", ascending=False
)

perturbation_rows = []
continuous_features = [
    feature
    for feature in permutation_features
    if X_robust[feature].nunique(dropna=True) > 10
][:6]

for feature in continuous_features:
    step = X_robust[feature].std() * 0.10
    if not np.isfinite(step) or step == 0:
        continue
    minus = X_robust.copy()
    plus = X_robust.copy()
    minus[feature] = minus[feature] - step
    plus[feature] = plus[feature] + step

    if (
        TASK_TYPE.lower() == "classification"
        and hasattr(model, "predict_proba")
        and len(model.classes_) == 2
    ):
        class_index = (
            list(model.classes_).index(POSITIVE_CLASS)
            if POSITIVE_CLASS in model.classes_
            else 1
        )
        base_output = model.predict_proba(X_robust)[:, class_index]
        minus_output = model.predict_proba(minus)[:, class_index]
        plus_output = model.predict_proba(plus)[:, class_index]
        sensitivity_measure = "mean_absolute_probability_change"
    else:
        base_output = model.predict(X_robust)
        minus_output = model.predict(minus)
        plus_output = model.predict(plus)
        sensitivity_measure = (
            "prediction_disagreement_rate"
            if TASK_TYPE.lower() == "classification"
            else "mean_absolute_prediction_change"
        )

    if TASK_TYPE.lower() == "classification" and sensitivity_measure == (
        "prediction_disagreement_rate"
    ):
        change = np.mean(
            (minus_output != base_output) | (plus_output != base_output)
        )
    else:
        change = np.mean(
            (
                np.abs(minus_output - base_output)
                + np.abs(plus_output - base_output)
            )
            / 2
        )
    perturbation_rows.append(
        {
            "feature": feature,
            "perturbation_step": step,
            "sensitivity_measure": sensitivity_measure,
            "mean_change": change,
        }
    )

perturbation_sensitivity = pd.DataFrame(perturbation_rows)
if not perturbation_sensitivity.empty:
    perturbation_sensitivity = perturbation_sensitivity.sort_values(
        "mean_change", ascending=False
    )

stability_features = feature_importance.head(15)["feature"].tolist()
stability_records = {feature: [] for feature in stability_features}
selection_counts = {feature: 0 for feature in stability_features}
for iteration in range(SHAP_STABILITY_ITERATIONS):
    positions = rng.integers(0, len(shap_frame), size=len(shap_frame))
    sampled_importance = (
        shap_frame.iloc[positions].abs().mean().sort_values(ascending=False)
    )
    ranking = {
        feature: rank + 1
        for rank, feature in enumerate(sampled_importance.index)
    }
    top_ten = set(sampled_importance.head(10).index)
    for feature in stability_features:
        stability_records[feature].append(ranking.get(feature, np.nan))
        if feature in top_ten:
            selection_counts[feature] += 1

stability_rows = []
for feature in stability_features:
    ranks = np.asarray(stability_records[feature], dtype=float)
    stability_rows.append(
        {
            "feature": feature,
            "mean_rank": np.nanmean(ranks),
            "rank_std": np.nanstd(ranks),
            "top_10_frequency": (
                selection_counts[feature] / SHAP_STABILITY_ITERATIONS
            ),
        }
    )
shap_rank_stability = pd.DataFrame(stability_rows).sort_values("mean_rank")

figures_round_4 = []
if not permutation_sensitivity.empty:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.barplot(
        data=permutation_sensitivity,
        y="feature",
        x="score_drop",
        color="#E15759",
        ax=ax,
    )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Decrease in held-out score after permutation")
    ax.set_ylabel("Transformed feature")
    ax.set_title("Held-out Permutation Sensitivity")
    figures_round_4.append(("01_permutation_sensitivity", fig))

if not perturbation_sensitivity.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(
        data=perturbation_sensitivity,
        y="feature",
        x="mean_change",
        color="#F28E2B",
        ax=ax,
    )
    ax.set_xlabel("Mean output change")
    ax.set_ylabel("Transformed feature")
    ax.set_title("Local Perturbation Sensitivity")
    figures_round_4.append(("02_perturbation_sensitivity", fig))

if not shap_rank_stability.empty:
    stability_plot = shap_rank_stability.sort_values(
        "mean_rank", ascending=False
    )
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.barplot(
        data=stability_plot,
        y="feature",
        x="top_10_frequency",
        color="#2CA25F",
        ax=ax,
    )
    for patch, row in zip(
        ax.patches, stability_plot.itertuples(index=False)
    ):
        ax.text(
            min(patch.get_width() + 0.015, 0.92),
            patch.get_y() + patch.get_height() / 2,
            f"mean rank {row.mean_rank:.1f}",
            va="center",
            fontsize=8,
        )
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Frequency selected in bootstrap top 10")
    ax.set_ylabel("Feature")
    ax.set_title("SHAP Importance Rank Stability")
    figures_round_4.append(("03_shap_rank_stability", fig))

round_4_dir, round_4_figures = save_round_bundle(4, figures_round_4)
round_4_tables = []
for name, table in [
    ("permutation_sensitivity.csv", permutation_sensitivity),
    ("perturbation_sensitivity.csv", perturbation_sensitivity),
    ("shap_rank_stability.csv", shap_rank_stability),
]:
    path = round_4_dir / name
    table.to_csv(path, index=False)
    round_4_tables.append(path)

most_sensitive_feature = (
    permutation_sensitivity.iloc[0].to_dict()
    if not permutation_sensitivity.empty
    else None
)
least_stable_top_feature = (
    shap_rank_stability.sort_values(
        ["top_10_frequency", "rank_std"],
        ascending=[True, False],
    ).iloc[0].to_dict()
    if not shap_rank_stability.empty
    else None
)
controller_reason_4 = (
    "This final planned round tests whether the explanation survives held-out "
    "feature permutation, small input perturbations, and repeated SHAP "
    "resampling. Further refinement should occur only when the LLM reviewer "
    "identifies a specific unresolved contradiction supported by these files."
)
round_4_evidence = {
    "held_out_sample_size": len(X_robust),
    "baseline_held_out_score": baseline_score,
    "score_definition": (
        "balanced_accuracy"
        if TASK_TYPE.lower() == "classification"
        else "r2"
    ),
    "most_permutation_sensitive_feature": most_sensitive_feature,
    "least_stable_reported_top_feature": least_stable_top_feature,
}
round_4_record = write_refinement_materials(
    4,
    REFINEMENT_ROUNDS[4],
    round_4_evidence,
    round_4_figures,
    round_4_tables,
    controller_reason_4,
    "conditional_stop_or_targeted_follow_up",
)

manifest_path = Path(OUTPUT_DIR) / "workflow_manifest.json"
manifest_path.write_text(
    json.dumps(workflow_manifest, indent=2, default=json_default),
    encoding="utf-8",
)

summary_path = Path(OUTPUT_DIR) / "refinement_summary.md"
summary_lines = [
    "# Generalized Agentic XAI Refinement Summary",
    "",
    f"Domain context: {DOMAIN_CONTEXT}",
    "",
    "## Completed Rounds",
    "",
    "- Round 0: baseline model, SHAP explanation, diagnostics, and reference comparison.",
]
for record in workflow_manifest:
    summary_lines.extend(
        [
            f"- Round {record['round']}: {record['analysis_module']}.",
            f"  Controller reason: {record['controller_reason']}",
        ]
    )
summary_lines.extend(
    [
        "",
        "## Interpretation Rule",
        "",
        "The workflow may stop after Round 4 unless the LLM self-reflection identifies "
        "a specific unresolved gap grounded in the generated evidence. Manual LLM mode "
        "creates prompts and recommendation drafts but does not invent an LLM response.",
        "",
        "## Reproducibility Warning",
        "",
        "Use the same dataset split, random seed, model parameters, LLM model, prompt "
        "settings, and human evaluation rubric when comparing recommendation rounds.",
    ]
)
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

display(permutation_sensitivity)
display(perturbation_sensitivity)
display(shap_rank_stability.head(15))
print(controller_reason_4)
print(f"Saved Round 4 to: {round_4_dir}")
print(f"Saved workflow manifest: {manifest_path}")
print(f"Saved refinement summary: {summary_path}")


,feature,baseline_score,permuted_score_mean,score_drop,score_drop_std
1,date_time_cos,0.605895,0.166491,0.439404,0.014349
0,date_time_sin,0.605895,0.175796,0.430099,0.005669
3,date_week_sin,0.605895,0.418748,0.187146,0.015078
4,Press_mm_hg,0.605895,0.511594,0.094301,0.009667
6,RH_3,0.605895,0.527272,0.078623,0.000253
2,lights,0.605895,0.532534,0.073360,0.011882
5,T3,0.605895,0.537108,0.068787,0.008132
7,T8,0.605895,0.553077,0.052818,0.000664
9,RH_1,0.605895,0.564430,0.041464,0.000685
8,T5,0.605895,0.565840,0.040055,0.002009


,feature,perturbation_step,sensitivity_measure,mean_change
0,date_time_sin,0.070827,mean_absolute_prediction_change,7.317533
1,date_time_cos,0.070607,mean_absolute_prediction_change,3.139092
2,date_week_sin,0.070081,mean_absolute_prediction_change,2.993408
3,Press_mm_hg,0.751360,mean_absolute_prediction_change,2.856267
4,T3,0.196043,mean_absolute_prediction_change,1.882083
5,RH_3,0.329819,mean_absolute_prediction_change,1.834500


,feature,mean_rank,rank_std,top_10_frequency
0,date_time_sin,1.00,0.000000,1.00
1,date_time_cos,2.00,0.000000,1.00
2,lights,3.04,0.195959,1.00
3,date_week_sin,3.96,0.195959,1.00
4,Press_mm_hg,5.46,0.498397,1.00
5,T3,5.55,0.517204,1.00
6,RH_3,7.01,0.172916,1.00
7,T8,8.40,0.678233,1.00
8,T5,8.92,0.702567,0.99
9,RH_1,10.43,1.159784,0.60


This final planned round tests whether the explanation survives held-out feature permutation, small input perturbations, and repeated SHAP resampling. Further refinement should occur only when the LLM reviewer identifies a specific unresolved contradiction supported by these files.
Saved Round 4 to: Evidence\Round_4
Saved workflow manifest: Evidence\workflow_manifest.json
Saved refinement summary: Evidence\refinement_summary.md
